# Model development

Final notebook that assumes all insights from `EDA.ipynb` and `Baseline.ipynb` to develop production model

## Download data

In [ ]:
import pandas as pd 
import numpy as np

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [2]:
%pip install -q kagglehub

In [6]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)
df = pd.read_csv(path)
df.head()

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data preparation

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()


# Drop customerId column as never significant
df = df.drop(columns="customerID")
df.shape

print("Basic data preparation is done")
print(df.shape)

print("Apply OHE for data")
df_ohe = pd.get_dummies(df)
print(df_ohe.shape)

Data preparation is done
(7032, 20)
Apply OHE for data
(7032, 46)


## Model building

I will use a GBDT approach as one of the most powerful in `Baseline.ipynb`

In [8]:
%pip install lightgbm -q

In [10]:
import lightgbm as lgb
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

churn_ratio = df_ohe.groupby("Churn")["Churn"].count()

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

minor_base_scale = churn_ratio.iloc[0] / churn_ratio.iloc[1]
lgbm = lgb.LGBMClassifier(scale_pos_weight=minor_base_scale, verbose=-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

y_pred_cv = cross_val_predict(lgbm, X, y, cv=skf)
y_proba_cv = cross_val_predict(lgbm, X, y, cv=skf, method="predict_proba")[:, 1]
auc_scores = cross_val_score(lgbm, X, y, cv=skf, scoring='average_precision')
print(classification_report(y, y_pred_cv))
print(f"PR-AUC: {auc_scores.mean():.4f} +/- {auc_scores.std():.4f} (var: {auc_scores.var():.4f})")

              precision    recall  f1-score   support

       False       0.89      0.76      0.82      5163
        True       0.53      0.75      0.62      1869

    accuracy                           0.76      7032
   macro avg       0.71      0.75      0.72      7032
weighted avg       0.80      0.76      0.77      7032

PR-AUC: 0.6533 +/- 0.0139 (var: 0.0002)
